# IndoBERTweet fine-tuning on cleaned text, then CatBoost on the fine-tuned embeddings

In [ ]:
DATA_DIR = "../data"

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/penyisihan-bdc-2024/sample_submission.csv
/kaggle/input/penyisihan-bdc-2024/val_v2.csv
/kaggle/input/penyisihan-bdc-2024/trainaug_v2.csv
/kaggle/input/penyisihan-bdc-2024/train_cleaned_v2_translated.csv
/kaggle/input/penyisihan-bdc-2024/trainaug_indo.csv
/kaggle/input/penyisihan-bdc-2024/train_id.csv
/kaggle/input/penyisihan-bdc-2024/val.csv
/kaggle/input/penyisihan-bdc-2024/val_indo.csv
/kaggle/input/penyisihan-bdc-2024/train_cleaned_v2_keywords.csv
/kaggle/input/penyisihan-bdc-2024/test_bdc.xlsx
/kaggle/input/penyisihan-bdc-2024/trainaug.csv
/kaggle/input/penyisihan-bdc-2024/train_eng_v2.csv
/kaggle/input/penyisihan-bdc-2024/train_bdc.xlsx


In [2]:
import pandas as pd
import numpy as np
import re

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)

import tensorflow as tf
from catboost import CatBoostClassifier, Pool

from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score, \
    balanced_accuracy_score
from datasets import Dataset, DatasetDict

2024-06-11 19:17:01.056265: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-11 19:17:01.056412: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-11 19:17:01.185746: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [18]:
import torch
print(torch.cuda.is_available())

True


In [5]:
train = pd.read_csv(f'{DATA_DIR}/train_cleaned_v2_keywords.csv')

In [6]:
train.head()

,text,label,pendidikan rendah,kelompok etnis,masyarakat lintas etnis,kaum difabel,bugis,generasi emas,low educated,menengah bawah,...,nyerocos,manyargubi,politik merdeka,futrijasun,vidaanasution,imadarch,chairudinn6548,sering ngibul,tata kata,ryan_fighter7
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Anies dapat tepuk tangan meriah saat jadi Rekt...,Politik,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,emng bener sih pendukung 01 ada yang goblok be...,Demografi,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Sewaktu anies bersikap kritis ke kinerja pak p...,Politik,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Anies Baswedan Harap aparatur sipil negara ter...,Politik,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
# Create a label mapping
label_mapping = {
    'Politik': 0,
    'Sosial Budaya': 1,
    'Ekonomi': 2,
    'Pertahanan dan Keamanan': 3,
    'Ideologi': 4,
    'Sumber Daya Alam': 5,
    'Demografi': 6,
    'Geografi': 7
}

# Apply the label mapping
train['label'] = train['label'].map(label_mapping)

In [8]:
# Split the DataFrame into training and validation sets
train_df, val_df = train_test_split(train[['text','label']], test_size=0.2, stratify=train['label'], random_state=42)

# Convert DataFrames to Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Combine into a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

In [9]:
import torch
import gc


# Mengosongkan cache CUDA
torch.cuda.empty_cache()

# Mengumpulkan garbage
gc.collect()

# Memeriksa penggunaan memori CUDA
print(f"Memory Reserved: {torch.cuda.memory_reserved()} bytes")
print(f"Memory Allocated: {torch.cuda.memory_allocated()} bytes")
print(f"Max Memory Reserved: {torch.cuda.max_memory_reserved()} bytes")

# Restart kernel/environment jika diperlukan

Memory Reserved: 0 bytes
Memory Allocated: 0 bytes
Max Memory Reserved: 0 bytes


In [14]:
import numpy as np
from collections import Counter

# Assuming 'dataset' is your dataset and 'train' split contains the labels
labels = dataset['train']['label']
class_counts = Counter(labels)
total_samples = len(labels)

# Calculate weights for each class
class_weights = {cls: total_samples/count for cls, count in class_counts.items()}

# Normalize weights to sum to 1
sum_weights = sum(class_weights.values())
class_weights = {cls: weight/sum_weights for cls, weight in class_weights.items()}

# Convert weights to a list
class_weight_list = [class_weights[i] for i in range(len(class_weights))]

In [15]:
class_weight_list

[0.0036310738871766324,
 0.026910557459897868,
 0.03877160885217513,
 0.03971266731946094,
 0.04049905677133145,
 0.08020401439028384,
 0.18592748790474892,
 0.5843435334149252]

In [16]:
# Define the model name
model_name = "indolem/indobertweet-base-uncased"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define the tokenize function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Tokenize the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)

# Define a data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/3336 [00:00<?, ? examples/s]

Map:   0%|          | 0/834 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indolem/indobertweet-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [24]:
import torch
from transformers import Trainer, TrainingArguments
from sklearn.metrics import balanced_accuracy_score

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Convert class weights to a tensor
        class_weights_tensor = torch.tensor(class_weight_list).to(model.parameters().__next__().device)

        # Compute weighted cross-entropy loss
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

# Define training arguments with evaluation and saving steps
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="steps",
    save_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=10,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)
    

# Use the custom trainer for training
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
)

# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

# Clear CUDA cache if necessary
torch.cuda.empty_cache()

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss,Validation Loss
100,1.738000,1.700014
200,1.541500,1.324953
300,1.179100,1.277441
400,1.019400,1.253477
500,0.962800,1.210857
600,1.054400,1.223467
700,0.848000,1.250918
800,0.738200,1.262628
900,0.557500,1.300358
1000,0.541300,1.315059


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead u

Balanced Accuracy: 0.5167603208554771


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import balanced_accuracy_score

# Define the model name
model_name = "indolem/indobertweet-base-uncased"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define the tokenize function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Tokenize the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)

# Define a data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Define training arguments with evaluation and saving steps
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="steps",
    save_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=10,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
)

# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

In [ ]:
from tqdm import tqdm

model = AutoModelForSequenceClassification.from_pretrained('results/checkpoint-500', num_labels=8)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Function to get embeddings using the fine-tuned model
def get_embeddings(texts, tokenizer, model, max_length=512):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        
    # Use the embeddings from the last hidden state (average pooling)
    embeddings = outputs.hidden_states[-1].mean(dim=1).cpu().numpy()
    return embeddings

# Assuming your train dataset is in a pandas DataFrame
train_texts = train['text'].tolist()

# Get embeddings for the train dataset
train_embeddings = []
for text in tqdm(train_texts, desc="Generating embeddings"):
    embedding = get_embeddings([text], tokenizer, model)
    train_embeddings.append(embedding)

train_embeddings = np.vstack(train_embeddings)

# Convert embeddings to DataFrame and add as new features to the original dataset
embedding_df = pd.DataFrame(train_embeddings)
train_dataset_with_embeddings = pd.concat([train, embedding_df], axis=1)

# Print the shape to verify
print(train_dataset_with_embeddings.shape)

In [ ]:
train_dataset_with_embeddings.to_csv(f'{DATA_DIR}/train_dataset_with_embeddings.csv', index=False)

In [ ]:
train_dataset_with_embeddings

In [ ]:
train_dataset_with_embeddings

In [ ]:
# Split the DataFrame into training and validation sets
train_df_split, val_df_split = train_test_split(train_dataset_with_embeddings, test_size=0.2, stratify=train_dataset_with_embeddings['label'], random_state=42)

In [ ]:
train_df_split.drop('label', axis=1).columns

In [ ]:
class_weights_dict = dict()
n, num_class = train_df_split.shape[0], len(train_df_split['label'].unique())
for label in train_df_split['label'].unique():
    n_l = train_df_split[train_df_split['label'] == label].shape[0]
    class_weights_dict[label] = n / (num_class * n_l)

class_weights_dict

In [ ]:
train_df_split.reset_index(drop=True, inplace=True)
val_df_split.reset_index(drop=True, inplace=True)

train_pool = Pool(
    # data=train_df_split.drop("label", axis=1).to_numpy(),
    data=train_df_split.drop(['label','text'], axis=1).to_numpy(),
    label=train_df_split['label'].to_numpy(),
    feature_names=train_df_split.drop(['label','text'], axis=1).columns.to_list()
)

val_pool = Pool(
    # data=val_df_split.drop("label", axis=1).to_numpy(),
    data=val_df_split.drop(['label','text'], axis=1).to_numpy(),
    label=val_df_split['label'].to_numpy(),
    feature_names=val_df_split.drop(['label','text'], axis=1).columns.to_list()
)

In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights=class_weights_dict,
    iterations = 100,
    learning_rate = 0.01
)

# cb_model.fit(
#     train_pool,
# #     eval_set=val_pool
# )

In [ ]:
cv_method = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
cv_results_model_1 = cross_val_score(estimator=cb_model, X=train_dataset_with_embeddings.drop(["label",'text'], axis=1), y=train_dataset_with_embeddings["label"], cv=cv_method,
                                     verbose=100, scoring='balanced_accuracy')

print("Model 1 - Balanced Accuracy scores: ")
print(cv_results_model_1)
print("Mean: ")
print(cv_results_model_1.mean().round(3))

In [ ]:
# Load the tokenizer from the local directory
local_model_path = 'results/checkpoint-500/'
tokenizer = AutoTokenizer.from_pretrained(local_model_path)

# Define the tokenize function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Tokenize the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model from the local directory
model = AutoModelForSequenceClassification.from_pretrained(local_model_path, num_labels=8, ignore_mismatched_sizes=True)

In [ ]:
import shutil
shutil.make_archive('results/checkpoint-x', 'zip', 'results/checkpoint-500')

In [ ]:
# Only BERT
predictions = cb_model.predict(val_pool)

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], predictions)
print("Balanced Accuracy:", balanced_acc)

In [ ]:
# Split the DataFrame into training and validation sets
train_df_split, val_df_split = train_test_split(train_dataset_with_embeddings, test_size=0.2, stratify=train_dataset_with_embeddings['label'], random_state=42)

train_df_split.reset_index(drop=True, inplace=True)
val_df_split.reset_index(drop=True, inplace=True)

train_pool = Pool(
    # data=train_df_split.drop("label", axis=1).to_numpy(),
    data=train_df_split.drop('label', axis=1).to_numpy(),
    label=train_df_split['label'].to_numpy(),
    text_features=['text'],
    feature_names=train_df_split.drop('label', axis=1).columns.to_list()
)

val_pool = Pool(
    # data=val_df_split.drop("label", axis=1).to_numpy(),
    data=val_df_split.drop('label', axis=1).to_numpy(),
    label=val_df_split['label'].to_numpy(),
    text_features=['text'],
    feature_names=val_df_split.drop('label', axis=1).columns.to_list()
)

In [ ]:
import torch
import gc


# Mengosongkan cache CUDA
torch.cuda.empty_cache()

# Mengumpulkan garbage
gc.collect()

# Memeriksa penggunaan memori CUDA
print(f"Memory Reserved: {torch.cuda.memory_reserved()} bytes")
print(f"Memory Allocated: {torch.cuda.memory_allocated()} bytes")
print(f"Max Memory Reserved: {torch.cuda.max_memory_reserved()} bytes")

# Restart kernel/environment jika diperlukan

In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights=class_weights_dict
)

cb_model.fit(
    train_pool,
    eval_set=val_pool
)

In [ ]:
# text + embedding
predictions = cb_model.predict(val_pool)

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], predictions)
print("Balanced Accuracy:", balanced_acc)